In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/ramen-ratings.csv")

In [3]:
df.head()

,Review #,Brand,Variety,Style,Country,Stars,Top Ten
0,2580,New Touch,T's Restaurant Tantanmen,Cup,Japan,3.75,NaN
1,2579,Just Way,Noodles Spicy Hot Sesame Spicy Hot Sesame Guan...,Pack,Taiwan,1,NaN
2,2578,Nissin,Cup Noodles Chicken Vegetable,Cup,USA,2.25,NaN
3,2577,Wei Lih,GGE Ramen Snack Tomato Flavor,Pack,Taiwan,2.75,NaN
4,2576,Ching's Secret,Singapore Curry,Pack,India,3.75,NaN


In [4]:
df["Variety"].sample(100)

866                                Sichuan Hot Spicy Fish
795           Instant Noodles Black Pepper Crab Mi Goreng
264                        Cup Noodle Spicy Curry Chicken
1212    Soba Teriyaki Noodles With Japanese Yakisoba S...
2204                            Noodle King Lobster Thick
                              ...                        
2057                         Artificial Hot & Sour Shrimp
1929                    Mi Segera Mi Sup Perisa Kari Ayam
2411                                        Seafood Party
1654                           VegeMee Vegetarian Flavour
2484                       Kung Fu Artificial Pork Flavor
Name: Variety, Length: 100, dtype: object

In [5]:
df.Variety.value_counts()

Variety
Beef                                              7
Chicken                                           7
Yakisoba                                          6
Artificial Chicken                                6
Vegetable                                         6
                                                 ..
Oh! Ricey Pho Ga                                  1
Veggie Noodle Black Sesame Noodle                 1
Nuudeli Liha Nudlar Kott                          1
Artificial Beef Instant Noodles With Soup Base    1
Tom Yum Chili Flavor                              1
Name: count, Length: 2413, dtype: int64

In [6]:
df.Brand.value_counts()

Brand
Nissin           381
Nongshim          98
Maruchan          76
Mama              71
Paldo             66
                ... 
Golden Wonder      1
Peyang             1
Sanrio             1
China Best         1
Westbrae           1
Name: count, Length: 355, dtype: int64

In [7]:
df.Country.value_counts()

Country
Japan            352
USA              323
South Korea      309
Taiwan           224
Thailand         191
China            169
Malaysia         156
Hong Kong        137
Indonesia        126
Singapore        109
Vietnam          108
UK                69
Philippines       47
Canada            41
India             31
Germany           27
Mexico            25
Australia         22
Netherlands       15
Myanmar           14
Nepal             14
Pakistan           9
Hungary            9
Bangladesh         7
Colombia           6
Brazil             5
Cambodia           5
Fiji               4
Holland            4
Poland             4
Finland            3
Sarawak            3
Sweden             3
Dubai              3
Ghana              2
Estonia            2
Nigeria            1
United States      1
Name: count, dtype: int64

In [8]:
df.isnull().sum()

Review #       0
Brand          0
Variety        0
Style          2
Country        0
Stars          0
Top Ten     2539
dtype: int64

In [9]:
# drop top ten column
df.drop(columns=["Top Ten"], inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2580 entries, 0 to 2579
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Review #  2580 non-null   int64 
 1   Brand     2580 non-null   object
 2   Variety   2580 non-null   object
 3   Style     2578 non-null   object
 4   Country   2580 non-null   object
 5   Stars     2580 non-null   object
dtypes: int64(1), object(5)
memory usage: 121.1+ KB


In [11]:
df["Style"].value_counts()

Style
Pack    1531
Bowl     481
Cup      450
Tray     108
Box        6
Can        1
Bar        1
Name: count, dtype: int64

In [12]:
# Dataset shape and basic info
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

Dataset shape: (2580, 6)
Columns: ['Review #', 'Brand', 'Variety', 'Style', 'Country', 'Stars']

Data types:
Review #     int64
Brand       object
Variety     object
Style       object
Country     object
Stars       object
dtype: object

Unique values per column:
Review #: 2580
Brand: 355
Variety: 2413
Style: 7
Country: 38
Stars: 51


In [13]:
# Convert Stars to numeric, handling 'Unrated' values
df["Stars_numeric"] = pd.to_numeric(df["Stars"], errors="coerce")
print(f"Missing values after conversion: {df['Stars_numeric'].isnull().sum()}")

# Check the distribution of numeric ratings
print(f"\nRatings distribution:")
print(df["Stars_numeric"].describe())

# Handle unrated entries - replace with median rating
median_rating = df["Stars_numeric"].median()
df["Stars_cleaned"] = df["Stars_numeric"].fillna(median_rating)
print(f"\nFilled missing ratings with median: {median_rating}")

Missing values after conversion: 3

Ratings distribution:
count    2577.000000
mean        3.654676
std         1.015331
min         0.000000
25%         3.250000
50%         3.750000
75%         4.250000
max         5.000000
Name: Stars_numeric, dtype: float64

Filled missing ratings with median: 3.75


In [14]:
# 1.1 One-Hot Encoding for Style column (categorical, low cardinality)
# Create a copy for feature engineering
df_fe = df.copy()

# One-hot encoding for Style (categorical variable)
style_dummies = pd.get_dummies(df_fe["Style"], prefix="Style")
df_fe = pd.concat([df_fe, style_dummies], axis=1)

print("Style One-Hot Encoding:")
print(f"Created {len(style_dummies.columns)} style dummy variables:")
print(list(style_dummies.columns))

Style One-Hot Encoding:
Created 7 style dummy variables:
['Style_Bar', 'Style_Bowl', 'Style_Box', 'Style_Can', 'Style_Cup', 'Style_Pack', 'Style_Tray']


In [15]:
# 1.2 CatBoost Encoding for Country (high cardinality categorical)
from category_encoders import CatBoostEncoder

# Initialize CatBoost encoder for Country
catboost_country = CatBoostEncoder()

# Fit and transform Country using target variable (Stars_cleaned)
df_fe["Country_catboost"] = catboost_country.fit_transform(df_fe["Country"], df_fe["Stars_cleaned"])

print("Country CatBoost Encoding (sample):")
country_encoding_sample = (
    df_fe[["Country", "Country_catboost"]]
    .drop_duplicates()
    .sort_values("Country_catboost", ascending=False)
)
print(country_encoding_sample.head(10))

print(
    f"\nCountry encoding range: {df_fe['Country_catboost'].min():.3f} to {df_fe['Country_catboost'].max():.3f}"
)

Country CatBoost Encoding (sample):
      Country  Country_catboost
534  Malaysia          4.710319
281  Malaysia          4.706848
530  Malaysia          4.689628
542  Malaysia          4.685576
537  Malaysia          4.665924
489  Malaysia          4.665753
203  Malaysia          4.664970
101  Malaysia          4.663697
638  Malaysia          4.663140
615  Malaysia          4.650184

Country encoding range: 1.934 to 4.710


In [16]:
# 1.3 CatBoost Encoding for Brand (high cardinality categorical)
# Initialize CatBoost encoder for Brand
catboost_brand = CatBoostEncoder()

# Fit and transform Brand using target variable (Stars_cleaned)
df_fe["Brand_catboost"] = catboost_brand.fit_transform(df_fe["Brand"], df_fe["Stars_cleaned"])

print("Brand CatBoost Encoding (top 10 by encoding value):")
brand_encoding_sample = (
    df_fe[["Brand", "Brand_catboost"]]
    .drop_duplicates()
    .sort_values("Brand_catboost", ascending=False)
)
print(brand_encoding_sample.head(10))

print(
    f"\nBrand encoding range: {df_fe['Brand_catboost'].min():.3f} to {df_fe['Brand_catboost'].max():.3f}"
)

# Show some brand statistics
brand_counts = df_fe["Brand"].value_counts()
print(f"\nBrand distribution:")
print(f"Total unique brands: {df_fe['Brand'].nunique()}")
print(f"Top 5 most frequent brands:")
print(brand_counts.head())

Brand CatBoost Encoding (top 10 by encoding value):
        Brand  Brand_catboost
757   MyKuali        4.906164
732   MyKuali        4.900299
714   MyKuali        4.893652
1493  MyKuali        4.891866
1135  MyKuali        4.887165
615   MyKuali        4.886056
1080  MyKuali        4.882036
554   MyKuali        4.877708
599   MyKuali        4.877291
963   MyKuali        4.876418

Brand encoding range: 1.302 to 4.906

Brand distribution:
Total unique brands: 355
Top 5 most frequent brands:
Brand
Nissin      381
Nongshim     98
Maruchan     76
Mama         71
Paldo        66
Name: count, dtype: int64


In [17]:
# 2.1 Text Features from Variety column
import re

# Basic text features
df_fe["Variety_length"] = df_fe["Variety"].str.len()
df_fe["Variety_word_count"] = df_fe["Variety"].str.split().str.len()

# Check for specific flavor keywords
spicy_keywords = [
    "spicy",
    "hot",
    "chili",
    "chilli",
    "pepper",
    "fire",
    "volcano",
    "hell",
]
seafood_keywords = ["seafood", "shrimp", "prawn", "crab", "fish", "ocean"]
chicken_keywords = ["chicken", "poultry"]
beef_keywords = ["beef", "meat"]
vegetable_keywords = ["vegetable", "veggie", "mushroom"]

# Create binary features for flavor categories
df_fe["is_spicy"] = (
    df_fe["Variety"].str.lower().str.contains("|".join(spicy_keywords), na=False).astype(int)
)
df_fe["is_seafood"] = (
    df_fe["Variety"].str.lower().str.contains("|".join(seafood_keywords), na=False).astype(int)
)
df_fe["is_chicken"] = (
    df_fe["Variety"].str.lower().str.contains("|".join(chicken_keywords), na=False).astype(int)
)
df_fe["is_beef"] = (
    df_fe["Variety"].str.lower().str.contains("|".join(beef_keywords), na=False).astype(int)
)
df_fe["is_vegetable"] = (
    df_fe["Variety"].str.lower().str.contains("|".join(vegetable_keywords), na=False).astype(int)
)

print("Flavor category distribution:")
flavor_columns = ["is_spicy", "is_seafood", "is_chicken", "is_beef", "is_vegetable"]
for col in flavor_columns:
    print(f"{col}: {df_fe[col].sum()} ({df_fe[col].mean():.2%})")

Flavor category distribution:
is_spicy: 412 (15.97%)
is_seafood: 301 (11.67%)
is_chicken: 328 (12.71%)
is_beef: 240 (9.30%)
is_vegetable: 128 (4.96%)


In [18]:
# 2.2 SBERT Encoding for Variety column (semantic text features)
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained SBERT model (optimized for semantic similarity)
print("Loading SBERT model...")
model = SentenceTransformer("all-MiniLM-L6-v2")  # Lightweight, fast model
print("SBERT model loaded successfully")

# Clean variety text (handle missing values)
variety_texts = df_fe["Variety"].fillna("unknown").tolist()

print(f"Encoding {len(variety_texts)} variety descriptions...")
# Generate embeddings for all variety descriptions
variety_embeddings = model.encode(variety_texts, show_progress_bar=True)

print(f"Generated embeddings shape: {variety_embeddings.shape}")
print(f"Each variety description is represented by {variety_embeddings.shape[1]} dimensions")

# Add SBERT embeddings as features (using PCA to reduce dimensionality)
from sklearn.decomposition import PCA

# Reduce to top 10 principal components for manageable feature count
pca = PCA(n_components=10, random_state=42)
variety_pca = pca.fit_transform(variety_embeddings)

# Add PCA components as features
for i in range(10):
    df_fe[f"variety_sbert_pc{i + 1}"] = variety_pca[:, i]

print(
    f"\nAdded 10 SBERT PCA features explaining {pca.explained_variance_ratio_.sum():.3f} of variance"
)
print("SBERT PCA variance explained by component:")
for i, var_ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i + 1}: {var_ratio:.3f} ({var_ratio * 100:.1f}%)")

Loading SBERT model...
SBERT model loaded successfully
Encoding 2580 variety descriptions...


Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Generated embeddings shape: (2580, 384)
Each variety description is represented by 384 dimensions

Added 10 SBERT PCA features explaining 0.376 of variance
SBERT PCA variance explained by component:
  PC1: 0.104 (10.4%)
  PC2: 0.049 (4.9%)
  PC3: 0.039 (3.9%)
  PC4: 0.033 (3.3%)
  PC5: 0.031 (3.1%)
  PC6: 0.030 (3.0%)
  PC7: 0.026 (2.6%)
  PC8: 0.023 (2.3%)
  PC9: 0.022 (2.2%)
  PC10: 0.021 (2.1%)
